# Task 9: Network Resilience & Cascade Simulation Engine
## Build the person network and the clan network

In [1]:
import random
import numpy as np
import pandas as pd
import networkx as nx

S = "../data/staging/"
C = "../data/curated/"
people = pd.read_parquet(S + "politicians.parquet")
edges = pd.read_parquet(S + "kinship_edges.parquet")
clans = pd.read_parquet(S + "clans.parquet")
alliances = pd.read_parquet(S + "clan_alliances.parquet")
town = pd.read_parquet(C + "town_dynasty_stronghold")
officials = pd.read_parquet(C + "official_terms_dynastic")
town["year"] = town["year"].astype(int)            # partition folders come back as a category; make it a number
officials["year"] = officials["year"].astype(int)

LATEST = 2025
LOCAL = ["MAYOR", "VICE MAYOR", "COUNCILOR"]
info = people.set_index("person_uid")

# Person network: strong + medium kinship links, without same-name "twins" (same rule as Task 8)
sm = edges[edges["tier"].isin(["strong", "medium"])]
A, B = info.loc[sm["person_a"]], info.loc[sm["person_b"]]
twin = ((A["last_key"].to_numpy() == B["last_key"].to_numpy()) & (A["first_key"].to_numpy() == B["first_key"].to_numpy())
        & (A["middle_key"].isna().to_numpy() | B["middle_key"].isna().to_numpy() | (A["middle_key"].to_numpy() == B["middle_key"].to_numpy())))
sm = sm[~twin]
G = nx.Graph()
G.add_edges_from(zip(sm["person_a"], sm["person_b"]))

# Clan network: clans = nodes, alliances (maternal links between clans) = edges
CG = nx.Graph()
CG.add_nodes_from(clans["clan_id"])
CG.add_edges_from(zip(alliances["clan_a"], alliances["clan_b"]))

pieces = sorted((len(c) for c in nx.connected_components(G)), reverse=True)
print(f"person network: {G.number_of_nodes():,} people, {G.number_of_edges():,} links, {len(pieces):,} separate groups, largest = {pieces[0]}")
print(f"average relatives per person: {2 * G.number_of_edges() / G.number_of_nodes():.1f}")
allied = [c for c in CG if CG.degree(c) > 0]
print(f"clan network: {CG.number_of_nodes():,} clans, {CG.number_of_edges():,} alliances, {len(allied):,} clans have at least one ally")
o = officials[officials["year"].eq(LATEST)]
print(f"{LATEST}: {len(o):,} officials, {o['clan_id'].notna().sum():,} belong to a clan, {o['clan_id'].nunique():,} clans hold seats")

person network: 54,768 people, 96,327 links, 14,907 separate groups, largest = 267
average relatives per person: 3.5
clan network: 16,481 clans, 1,649 alliances, 2,765 clans have at least one ally
2025: 17,819 officials, 11,108 belong to a clan, 8,656 clans hold seats


A "family bloc" is a group of 5 or more connected relatives. We remove people from the network and measure what share of people are still inside a bloc. We try two ways:

Random: remove people at random (like ordinary election losses).
Targeted: remove the most-connected people first (the family "hubs", like a patriarch who links many relatives).

The robustness index R is the average bloc coverage across all steps. 1.0 means unbreakable, and lower means more fragile.

# Attack test (random vs targeted removal)

In [2]:
def bloc_coverage(H, min_size=5):
    """People who are still in a family bloc of 5+ connected relatives."""
    return sum(len(c) for c in nx.connected_components(H) if len(c) >= min_size)

def largest_bloc(H):
    return max((len(c) for c in nx.connected_components(H)), default=0)

N = G.number_of_nodes()
base = bloc_coverage(G)
by_degree = [n for n, _ in sorted(G.degree, key=lambda x: -x[1])]     # most-connected people first
fractions = [0, 0.01, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30]

rows = []
for strategy in ["random", "targeted (most-connected first)"]:
    for f in fractions:
        k = int(f * N)
        if strategy == "random":
            results = []
            for seed in range(3):                                   # average of 3 random runs
                H = G.copy()
                H.remove_nodes_from(random.Random(seed).sample(list(G.nodes), k))
                results.append((bloc_coverage(H), largest_bloc(H)))
            cov, big = np.mean(results, axis=0)
        else:
            H = G.copy()
            H.remove_nodes_from(by_degree[:k])
            cov, big = bloc_coverage(H), largest_bloc(H)
        rows.append({"strategy": strategy, "fraction_removed": f, "people_removed": k,
                     "bloc_coverage": round(cov / base, 3), "largest_bloc": int(big)})
attack_curves = pd.DataFrame(rows)
display(attack_curves.pivot(index="fraction_removed", columns="strategy", values="bloc_coverage"))
robustness = attack_curves.groupby("strategy")["bloc_coverage"].mean().round(3)
print("robustness index R (1 = unbreakable, lower = more fragile):")
print(robustness.to_string())

strategy,random,targeted (most-connected first)
fraction_removed,,
0.00,1.000,1.000
0.01,0.981,0.965
0.02,0.961,0.933
0.05,0.906,0.832
0.10,0.817,0.664
0.15,0.730,0.487
0.20,0.647,0.316
0.30,0.488,0.056


robustness index R (1 = unbreakable, lower = more fragile):
strategy
random                             0.816
targeted (most-connected first)    0.657


# Targeted clan disruption and cascade simulation

In [3]:
def cascade(seed_clans, theta=0.5):
    """Watts threshold cascade on the clan alliance network.
    A clan falls when at least `theta` of its allies have fallen. Returns (fallen clans, rounds)."""
    fallen, rounds = set(seed_clans), 0
    while True:
        new = {c for f in fallen for c in CG[f] if c not in fallen
               and sum(n in fallen for n in CG[c]) / CG.degree(c) >= theta}
        if not new:
            return fallen, rounds
        fallen |= new
        rounds += 1

towns_now = town[town["year"].eq(LATEST)].set_index(["province_std", "town_std"])

def impact(fallen):
    """Seats vacated by fallen clans, and what happens to town strongholds (seats go to non-dynastic winners)."""
    hit = o[o["clan_id"].isin(fallen)]
    lost = (hit[hit["position"].isin(LOCAL)].dropna(subset=["town_std"])
            .groupby(["province_std", "town_std"])["is_dynastic"].sum().rename("lost_dynastic"))
    tt = towns_now.join(lost, how="left").fillna({"lost_dynastic": 0})
    new_share = (tt["n_dynastic"] - tt["lost_dynastic"]) / tt["n_officials"]
    return {"officials_removed": hit["person_uid"].nunique(),
            "mayors_removed": int(hit["position"].eq("MAYOR").sum()),
            "governors_removed": int(hit["position"].eq("GOVERNOR").sum()),
            "stronghold_towns_after": int((new_share >= 0.75).sum()),
            "avg_town_share_after": round(float(new_share.mean()), 3)}

# Seats each clan holds in 2025, and the biggest clan in each province
seats = (o.dropna(subset=["clan_id"]).groupby("clan_id")["person_uid"].nunique()
           .reset_index(name="n").sort_values(["n", "clan_id"], ascending=[False, True]).set_index("clan_id")["n"])  # ties broken by clan_id
top_per_province = (o.dropna(subset=["clan_id"]).groupby(["province_std", "clan_id"])["person_uid"].nunique().rename("s").reset_index()
                    .sort_values(["s", "clan_id"], ascending=[False, True]).drop_duplicates("province_std"))

scenarios = {"no disruption (baseline)": []}
scenarios.update({f"top {k} clans by seats": list(seats.index[:k]) for k in [1, 10, 50, 100, 250]})
scenarios["top clan in every province"] = list(top_per_province["clan_id"].unique())
scenarios["250 random clans"] = random.Random(0).sample(list(seats.index), 250)

rows = []
for name, seed in scenarios.items():
    fallen, rounds = cascade(seed)
    row = {"scenario": name, "clans_targeted": len(seed), "clans_fell_by_cascade": len(fallen) - len(seed), "cascade_rounds": rounds}
    row.update(impact(fallen))
    rows.append(row)
scenario_results = pd.DataFrame(rows)
scenario_results

,scenario,clans_targeted,clans_fell_by_cascade,cascade_rounds,officials_removed,mayors_removed,governors_removed,stronghold_towns_after,avg_town_share_after
0,no disruption (baseline),0,0,0,0,0,0,519,0.649
1,top 1 clans by seats,1,1,1,29,5,0,517,0.647
2,top 10 clans by seats,10,15,4,140,27,3,508,0.642
3,top 50 clans by seats,50,74,4,464,84,17,474,0.626
4,top 100 clans by seats,100,102,4,704,138,27,443,0.615
5,top 250 clans by seats,250,193,4,1281,255,46,380,0.587
6,top clan in every province,87,72,4,531,101,29,470,0.625
7,250 random clans,250,62,3,392,48,3,456,0.628


# Clan Resilience Table

In [4]:
# Systemic importance: how many OTHER clans fall if only this clan is disrupted
reach = {c: len(cascade([c])[0]) - 1 for c in seats.index if CG.degree(c) > 0}

# Hub dependence: remove a clan's most-connected member; how much of the clan stays connected?
members = people.dropna(subset=["clan_id"]).groupby("clan_id")["person_uid"].apply(list)
def after_hub_removal(clan_id):
    sub = G.subgraph(members[clan_id]).copy()
    if sub.number_of_nodes() < 3:
        return np.nan                                   # too small to measure
    hub = max(sub.degree, key=lambda x: x[1])[0]
    sub.remove_node(hub)
    return round(largest_bloc(sub) / sub.number_of_nodes(), 3)

clan_resilience = pd.DataFrame({"clan_id": seats.index, "seats_2025": seats.values})
clan_resilience = clan_resilience.merge(clans[["clan_id", "clan_label", "n_members", "first_year", "last_year"]], on="clan_id", how="left")
clan_resilience["n_allies"] = clan_resilience["clan_id"].map(dict(CG.degree())).fillna(0).astype(int)
clan_resilience["cascade_reach"] = clan_resilience["clan_id"].map(reach).fillna(0).astype(int)
clan_resilience["connected_after_hub_removal"] = clan_resilience["clan_id"].map(after_hub_removal)
clan_resilience["years_in_power"] = clan_resilience["last_year"] - clan_resilience["first_year"]

print(f"clans with seats in {LATEST}: {len(clan_resilience):,}")
print(f"clans whose disruption topples at least one allied clan: {(clan_resilience['cascade_reach'] > 0).sum():,}")
print(f"median share of a clan still connected after losing its most-connected member: {clan_resilience['connected_after_hub_removal'].median():.0%}")
clan_resilience.head(15)

clans with seats in 2025: 8,656
clans whose disruption topples at least one allied clan: 1,483
median share of a clan still connected after losing its most-connected member: 100%


,clan_id,seats_2025,clan_label,n_members,first_year,last_year,n_allies,cascade_reach,connected_after_hub_removal,years_in_power
0,CLAN-00019,27,AMPATUAN | MAGUINDANAO DEL SUR,27,2025,2025,1,1,1.0,0
1,CLAN-00095,16,MIDTIMBANG | MAGUINDANAO DEL SUR,16,2025,2025,0,0,1.0,0
2,CLAN-00004,14,BALINDONG | LANAO DEL SUR,43,1995,2025,4,8,1.0,30
3,CLAN-00013,12,SINGSON | ILOCOS SUR,32,1984,2025,1,1,1.0,41
4,CLAN-00260,11,SANGKI | MAGUINDANAO DEL SUR,11,2025,2025,0,0,0.6,0
5,CLAN-00011,10,UY | ISABELA,33,2001,2025,1,1,1.0,24
6,CLAN-00343,10,SINSUAT | MAGUINDANAO DEL NORTE,10,2025,2025,0,0,1.0,0
7,CLAN-00005,9,SISON | PANGASINAN,39,1912,2025,3,3,1.0,113
8,CLAN-00015,9,DY | ISABELA,28,1992,2025,0,0,1.0,33
9,CLAN-00062,9,UY | ZAMBOANGA DEL NORTE,18,2001,2025,1,1,1.0,24


In [5]:
from pathlib import Path
out = Path(C)
for name, df in {"network_attack_curves": attack_curves,
                 "clan_disruption_scenarios": scenario_results,
                 "clan_resilience": clan_resilience}.items():
    df.to_parquet(out / f"{name}.parquet", index=False)        # overwrites each run (idempotent)
    print(f"{name:28s} {len(df):>6,} rows -> data/curated/{name}.parquet")

network_attack_curves            16 rows -> data/curated/network_attack_curves.parquet
clan_disruption_scenarios         8 rows -> data/curated/clan_disruption_scenarios.parquet
clan_resilience               8,656 rows -> data/curated/clan_resilience.parquet
